# Day 3 - Conversational AI - aka Chatbot!

In [ ]:
# import libraries

import os 
from dotenv import load_dotenv 
from openai import OpenAI
import gradio as gr


In [ ]:
load_dotenv(override=True)
gemini_api_key = os.getenv('GOOGLE_API_KEY')

if (gemini_api_key and gemini_api_key.startswith('AIz')):
    print(f'API key found and looks good so far')
else:
    print('API key not found')


In [ ]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(base_url=gemini_url, api_key=gemini_api_key)

gemini_2 = 'gemini-2.5-flash-lite'
gemini_3 = 'gemini-3.1-flash-lite'


## And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.


In [ ]:
def reply_mango(messages, history):
    return 'mangoes'


In [ ]:
app = gr.ChatInterface(fn=reply_mango) 
app.launch()

In [ ]:
def reply_mango_2(messages, history):
    return f'You said {messages}, and the history is {history}, but I still say mangoes'

## OK! Let's write a slightly better chat callback!

In [ ]:
# Again, I'll be in scientist-mode and change this global during the lab

system_message = "You are a helpful assistant"

In [ ]:
def chat(message, history):
    history = [{'role': h['role'], 'content': h['content']} for h in history]
    messages = [{'role': 'system', 'content': system_message}] + history + [{'role': 'user', 'content': message}]

    response = gemini.chat.completions.create(
        model=gemini_3,
        messages=messages
    )

    return response.choices[0].message.content


In [ ]:
app = gr.ChatInterface(fn=chat)
app.launch()

### Streaming the results

In [ ]:
def chat(message, history):
    history = [{'role': h['role'], 'content': h['content']} for h in history]
    messages = [{'role': 'system', 'content': system_message}] + history + [{'role': 'user', 'content': message}]
    streaming = gemini.chat.completions.create(
        model=gemini_3,
        messages=messages,
        stream=True
    )
    result = ''
    for chunk in streaming:
        result += chunk.choices[0].delta.content or ''
        yield result



In [ ]:
app = gr.ChatInterface(fn=chat)
app.launch()


## OK let's keep going!

Using a system message to add context, and to give an example answer.. this is "one shot prompting" again

In [ ]:
system_prompt = '''
You are a helpful assistant in a clothes store. You should try to gently encourage
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off.
For example, if the customer says 'I'm looking to buy a hat,
You could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.
Encourage the customer to buy hats if they are unsure what to get.
'''


In [ ]:
def chat(message, history):
    history = [{'role': h['role'], 'content': h['content']} for h in history]
    messages = [{'role': 'system', 'content': system_prompt}] + history + \
                [{'role':'user', 'content': message}]
    streaming = gemini.chat.completions.create(
        model=gemini_3,
        messages=messages,
        stream=True
    )
    response = ''
    for chunk in streaming:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
app = gr.ChatInterface(fn=chat)
app.launch()

In [ ]:
system_prompt += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [ ]:
app = gr.ChatInterface(fn=chat)
app.launch()

In [ ]:
def chat(message, history):
    history = [{'role': h['role'],'content': h['content']} for h in history]
    relevant_system_message = system_prompt
    if ('belt' in message.lower()):
        relevant_system_message += f'if the user asks for belts, mention the user that the store does not sells belts; instead point the user to other items in the sale.'

    messages = [{'role': 'system', 'content': relevant_system_message}] + history + \
                [{'role': 'user', 'content': message}]

    streaming = gemini.chat.completions.create(
        model=gemini_3,
        messages=messages,
        stream=True
    )
    response = ''
    for chunk in streaming:
        response += chunk.choices[0].delta.content or ''
        yield response


In [ ]:
app = gr.ChatInterface(fn=chat)
app.launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Conversational Assistants are of course a hugely common use case for Gen AI, and the latest frontier models are remarkably good at nuanced conversation. And Gradio makes it easy to have a user interface. Another crucial skill we covered is how to use prompting to provide context, information and examples.
<br/><br/>
Consider how you could apply an AI Assistant to your business, and make yourself a prototype. Use the system prompt to give context on your business, and set the tone for the LLM.</span>
        </td>
    </tr>
</table>